In [23]:
import torch
torch.set_printoptions(precision=4)

$$
X =
\begin{bmatrix}
1 & 0 & 2 & 1 \\
0 & 1 & 1 & 0 \\
2 & 1 & 0 & 1
\end{bmatrix},
\quad
h_0 = [0,0],
\quad
W_x =
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 1 \\
0 & 1
\end{bmatrix},
\quad
W_h =
\begin{bmatrix}
1 & 0 \\
1 & 1
\end{bmatrix},
\quad
W_o =
\begin{bmatrix}
1 \\
2
\end{bmatrix}
$$

In [24]:
x = torch.tensor([
    [[1., 0., 2., 1.],
     [0., 1., 1., 0.],
     [2., 1., 0., 1.]]
])

y = torch.tensor([[20.]])


Wx = torch.tensor([
    [1., 0.],
    [0., 1.],
    [1., 1.],
    [0., 1.]
], requires_grad=True)

Wh = torch.tensor([
    [1., 0.],
    [1., 1.]
], requires_grad=True)

Wo = torch.tensor([
    [1.],
    [2.]
], requires_grad=True)


h0 = torch.zeros(1, 2)

## Forward Pass 
$$
h_1 = x_1 W_x + h_0 W_h = [3,3] 
$$

$$
h_2 = x_2 W_x + h_1 W_h = [7,5]
$$

$$
h_3 = x_3 W_x + h_2 W_h = [14,7] 
$$

$$
\hat{y} = h_3 W_o = 28 
$$


In [36]:
h1 = x[:,0,:] @ Wx + h0 @ Wh 
h2 = x[:,1,:] @ Wx + h1 @ Wh
h3 = x[:,2,:] @ Wx + h2 @ Wh

y_hat = h3 @ Wo

loss = 0.5 * (y_hat - y)**2

loss.backward()

In [37]:
print('h1 : ',h1.data)
print('h2 : ',h2.data)
print('h3 : ',h3.data)
print('y_hat : ',y_hat.data.item())
print('loss : ',loss.data.item())

h1 :  tensor([[3., 3.]])
h2 :  tensor([[7., 5.]])
h3 :  tensor([[14.,  7.]])
y_hat :  28.0
loss :  32.0


## BackPropagation Through Time ( BPTT ) 

 * We are using squared loss function, $L = \frac{1}{2} (\hat{y} - y)^2$
 * Using this given $y=20$ and we get the loss as $L = \frac{1}{2}(28 - 20)^2 = 32$
 * First we compute the gradient with respect to the output $\frac{\partial L}{\partial \hat{y}} = (\hat{y} - y) = 8 $
 * **Gradient w.r.t output weights $W_o$**

$$
\frac{\partial L}{\partial W_o} =\frac{\partial L}{\partial \hat{y}} * \frac{\partial \hat{y}}{\partial W_o} = 8 * h_3 =
\begin{bmatrix}
14 \\
7
\end{bmatrix}
(8)
=
\begin{bmatrix}
112 \\
56
\end{bmatrix}
$$

 * Computing the gradient of Loss with respect to each hidden state ( t=3,2,1) 
$$
\frac{\partial L}{\partial h_3}
=
\frac{\partial L}{\partial \hat{y}} * \frac{\partial \hat{y}}{\partial h_3}
=
8 [1,2]
=
[8,16]
$$


$$
\frac{\partial L}{\partial h_2} =
\frac{\partial L}{\partial \hat{y}} * \frac{\partial \hat{y}}{\partial h_3} * \frac{\partial h_3}{\partial h_2}=[8,16]*W_h =[8,24]
$$

$$
\frac{\partial L}{\partial h_1} = [8,32]
$$

 * **Gradient w.r.t hidden weight matrix $W_h$**
$$
\frac{\partial L}{\partial W_h}
=
\frac{\partial L}{\partial h_3}\cdot\frac{\partial h_3}{\partial W_h}
+
\frac{\partial L}{\partial h_2}\cdot\frac{\partial h_2}{\partial W_h}
+
\frac{\partial L}{\partial h_1}\cdot\frac{\partial h_1}{\partial W_h}
$$

$$
\frac{\partial L}{\partial W_h}
=
\frac{\partial L}{\partial h_3}\cdot h_2^T
+
\frac{\partial L}{\partial h_2}\cdot h_1^T
+
\frac{\partial L}{\partial h_1}\cdot h_0^T
$$


$$
\frac{\partial L}{\partial W_h}
=
\begin{bmatrix}
7 \\
5
\end{bmatrix}
[8,16]
+
\begin{bmatrix}
3 \\
3
\end{bmatrix}
[24,16]
+
\begin{bmatrix}
0 \\
0
\end{bmatrix}
[40,16]
$$

$$
\frac{\partial L}{\partial W_h}
=
\begin{bmatrix}
80 & 184 \\
64 & 152
\end{bmatrix}
$$



* **Gradient w.r.t input weight matrix $W_x$**

$$
\frac{\partial L}{\partial W_x}
=
\frac{\partial L}{\partial h_3}\cdot\frac{\partial h_3}{\partial W_x}
+
\frac{\partial L}{\partial h_2}\cdot\frac{\partial h_2}{\partial W_x}
+
\frac{\partial L}{\partial h_1}\cdot\frac{\partial h_1}{\partial W_x}
$$

$$
\frac{\partial L}{\partial W_x}
=
\frac{\partial L}{\partial h_3}\cdot x_3^T
+
\frac{\partial L}{\partial h_2}\cdot x_2^T
+
\frac{\partial L}{\partial h_1}\cdot x_1^T
$$

$$
\frac{\partial L}{\partial W_x}
=
\begin{bmatrix}
8 \\
16
\end{bmatrix}
[2,1,0,1]
+
\begin{bmatrix}
24 \\
16
\end{bmatrix}
[0,1,1,0]
+
\begin{bmatrix}
40 \\
16
\end{bmatrix}
[1,0,2,1]
$$

$$
\frac{\partial L}{\partial W_x}
=
\begin{bmatrix}
24 & 64 \\
16 & 40 \\
24 & 88 \\
16 & 48
\end{bmatrix}
$$

In [41]:
print(Wh.grad)
print('-'*50)
print(Wx.grad)
print('-'*50)
print(Wo.grad)

tensor([[ 80., 184.],
        [ 64., 152.]])
--------------------------------------------------
tensor([[24., 64.],
        [16., 40.],
        [24., 88.],
        [16., 48.]])
--------------------------------------------------
tensor([[112.],
        [ 56.]])


$W_h$ is reused at every timestep: $h_1 \rightarrow W_h,\; h_2 \rightarrow W_h,\; h_3 \rightarrow W_h$

So the same parameter affects the loss through multiple independent paths, and therefore its gradient is the **sum of contributions from all timesteps**.

In [51]:
import torch
import torch.nn as nn

class RNN_M2O(nn.Module):
    # RNN Many to One Implementation 
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        # Weight matrices
        self.Wx = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.1)
        self.Wh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.1)
        self.Wo = nn.Parameter(torch.randn(hidden_dim, output_dim) * 0.1)

    def forward(self, x):
        """
        x shape: (batch, time, input_dim)
        """

        batch_size, seq_len, _ = x.shape

        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        self.hidden_states = []  # optional: for debugging

        for t in range(seq_len):
            x_t = x[:, t, :]

            # h_t = x_t W_x + h_{t-1} W_h
            h = x_t @ self.Wx + h @ self.Wh

            self.hidden_states.append(h)

        # many-to-one output using last hidden state
        y_hat = h @ self.Wo

        return y_hat

In [52]:
model = RNN_M2O(input_dim=4, hidden_dim=2, output_dim=1)

x = torch.tensor([[
    [1., 0., 2., 1.],
    [0., 1., 1., 0.],
    [2., 1., 0., 1.]
]])

y = torch.tensor([[20.]])

y_hat = model(x)

loss = 0.5 * (y_hat - y) ** 2
loss.backward()

print(model.Wh.grad)
print(model.Wx.grad)
print(model.Wo.grad)

tensor([[ 0.0676, -0.1613],
        [-0.2609,  0.4295]])
tensor([[-1.6889,  3.2039],
        [-0.9475,  1.5307],
        [-0.0908, -0.0867],
        [-0.8420,  1.5989]])
tensor([[-1.7508],
        [ 0.6746]])


In [53]:
y_hat

tensor([[0.0064]], grad_fn=<MmBackward0>)

In [54]:
model.hidden_states

[tensor([[0.1251, 0.2451]], grad_fn=<AddBackward0>),
 tensor([[-0.0947,  0.2789]], grad_fn=<AddBackward0>),
 tensor([[ 0.0876, -0.0337]], grad_fn=<AddBackward0>)]

In [59]:
import torch
import torch.nn as nn

class RNN_M2M(nn.Module):
    # RNN Many to Many Implementation 
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Shared parameters across time
        self.Wx = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.1)
        self.Wh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.1)
        self.Wo = nn.Parameter(torch.randn(hidden_dim, output_dim) * 0.1)

    def forward(self, x):
        """
        x shape: (batch, time, input_dim)
        returns: (batch, time, output_dim)
        """

        batch_size, seq_len, _ = x.shape

        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        outputs = []
        self.hidden_states = [] 

        for t in range(seq_len):
            x_t = x[:, t, :]

            # hidden update (linear RNN)
            h = x_t @ self.Wx + h @ self.Wh

            self.hidden_states.append(h)
            # output at each timestep
            y_t = h @ self.Wo

            outputs.append(y_t)

        # stack over time dimension
        outputs = torch.stack(outputs, dim=1)

        return outputs,torch.stack(self.hidden_states, dim=1)

In [61]:
model = RNN_M2M(input_dim=4, hidden_dim=2, output_dim=1)

x = torch.tensor([[
    [1., 0., 2., 1.],
    [0., 1., 1., 0.],
    [2., 1., 0., 1.]
]])

y = torch.tensor([[
    [10.],
    [15.],
    [20.]
]])

y_hat,hidden_states = model(x)

loss = 0.5 * ((y_hat - y) ** 2).mean()
loss.backward()

print("Wh grad:\n", model.Wh.grad)
print("Wx grad:\n", model.Wx.grad)
print("Wo grad:\n", model.Wo.grad)

Wh grad:
 tensor([[ 1.1188e-02, -7.0213e-04],
        [-1.5143e-04,  2.1478e-01]])
Wx grad:
 tensor([[ 0.3291,  3.1395],
        [ 0.1835,  2.2557],
        [-0.0527,  2.5022],
        [ 0.1502,  1.9330]])
Wo grad:
 tensor([[-0.8470],
        [-2.7693]])


In [62]:
y_hat

tensor([[[-0.0362],
         [-0.0006],
         [-0.0595]]], grad_fn=<StackBackward0>)

In [63]:
model.hidden_states

[tensor([[-0.0748,  0.2120]], grad_fn=<AddBackward0>),
 tensor([[ 0.0645, -0.0063]], grad_fn=<AddBackward0>),
 tensor([[0.1159, 0.3128]], grad_fn=<AddBackward0>)]